# 01 – Data Acquisition & Cleaning

This notebook pulls data from two APIs and saves two clean CSV files to `data/raw/`.
**The rest of the group only needs these CSV files. They don't need to run this notebook or have an API key.**

| File | One row per | Sorted by | Columns |
|---|---|---|---|
| `census_places.csv` | U.S. city/town | population, largest first | geoid, city, state, population, median_income, pct_bike_commute, pct_no_vehicle |
| `citybike_networks_us.csv` | U.S. bike-share network | number of stations, most first | network_id, name, city, state_abbr, n_stations, total_docks, pct_missing_capacity |

Load them with:
```python
census = pd.read_csv('data/raw/census_places.csv', dtype={'geoid': str})   # keeps geoid's leading zeros
bikes = pd.read_csv('data/raw/citybike_networks_us.csv')
```

## 1. Setup: imports, API key, output folder

The Census key is **never typed into this notebook**:
- **Colab:** in the Secrets panel (key icon), add a secret named `CENSUS_API_KEY`.
- **Local:** put `CENSUS_API_KEY=...` in `.env`, which is gitignored.

Don't print the key, because cell output gets saved into the notebook.

In [ ]:
import os
import time
from pathlib import Path

import requests
import pandas as pd

try:
    from google.colab import userdata              # Colab: key lives in the Secrets panel
    CENSUS_API_KEY = userdata.get('CENSUS_API_KEY')
except ImportError:
    from dotenv import load_dotenv                 # Local: key lives in .env (gitignored)
    load_dotenv()
    CENSUS_API_KEY = os.getenv('CENSUS_API_KEY')
assert CENSUS_API_KEY, 'No Census API key found - see README Setup'

# repo's data/raw when run from notebooks/, else ./data/raw (Colab - download the files before the session ends)
DATA_DIR = Path('../data/raw') if Path('../data').exists() else Path('data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 2. CityBikes: U.S. networks

`/v2/networks` lists every bike-share network in the world. We **keep only the U.S. ones**.
CityBikes city names look like `"Chicago, IL"`, so we split them into `city` and `state_abbr`.

In [ ]:
response_bikes = requests.get('https://api.citybik.es/v2/networks', timeout=30)
response_bikes.raise_for_status()

all_networks = pd.json_normalize(response_bikes.json()['networks'])      # flattens the nested "location" field
networks_df = (all_networks[all_networks['location.country'] == 'US']   # U.S. only
    .rename(columns={'id': 'network_id', 'location.city': 'location'})
    [['network_id', 'name', 'location']])
networks_df[['city', 'state_abbr']] = networks_df['location'].str.split(', ', n=1, expand=True)

print(len(all_networks), 'networks worldwide ->', len(networks_df), 'in the U.S.')

## 3. CityBikes: count stations and docks per network

We download every station in each U.S. network, waiting 1 second between networks so we don't overload the free API. This takes about a minute.

**Docks per station (`capacity`):** use the real dock count (`extra.slots`) if the station reports one, otherwise `free_bikes + empty_slots`.
Dockless systems (no `empty_slots`, or Lime's `999999` placeholder) get `NaN`, **not 0**.

Each network's results:
- `n_stations`: number of stations
- `total_docks`: total of `capacity` across its stations
- `pct_missing_capacity`: share of stations with no dock count (0–1). A value of 1.0 means the network is dockless, so its `total_docks` is blank.

In [ ]:
def fetch_stations(network_id):
    """Download all stations for one network; return [] on failure so one bad network doesn't stop the loop."""
    try:
        response = requests.get(f'https://api.citybik.es/v2/networks/{network_id}', timeout=30)
        if response.status_code == 200:
            stations = response.json()['network']['stations']
            for s in stations:
                s['network_id'] = network_id      # remember which network each station belongs to
            return stations
        print(network_id, 'failed:', response.status_code)
    except requests.RequestException as e:
        print(network_id, 'error:', e)
    return []


all_stations = []
for net_id in networks_df['network_id']:
    all_stations.extend(fetch_stations(net_id))
    time.sleep(1)                                 # be polite to the API

# reindex adds any column no station reported (e.g. extra.slots) as all-NaN instead of crashing
stations = pd.json_normalize(all_stations).reindex(
    columns=['network_id', 'id', 'free_bikes', 'empty_slots', 'extra.slots'])
empty_slots = stations['empty_slots'].mask(stations['empty_slots'] >= 999999)   # Lime's "no limit" placeholder
stations['capacity'] = stations['extra.slots'].fillna(stations['free_bikes'] + empty_slots)
assert not (stations['capacity'] > 1000).any(), 'a station reports 1000+ docks - probably a placeholder value'

citybike_df = (stations.groupby('network_id')
    .agg(n_stations=('id', 'count'),
         total_docks=('capacity', lambda c: c.sum(min_count=1)),   # all-NaN -> NaN, not 0
         pct_missing_capacity=('capacity', lambda c: c.isna().mean().round(4)))
    .reset_index()
    .merge(networks_df[['network_id', 'name', 'city', 'state_abbr']], on='network_id')
    [['network_id', 'name', 'city', 'state_abbr', 'n_stations', 'total_docks', 'pct_missing_capacity']]
    .sort_values('n_stations', ascending=False)
    .reset_index(drop=True))

citybike_df.head(10)

## 4. Census ACS 5-year (2024): every U.S. city

One request returns every city/town ("place") in every state. Variable codes come from
https://api.census.gov/data/2024/acs/acs5/variables.html.

If the request fails we only print the type of error. The `requests` error message contains the full URL, and the URL contains the key.

In [ ]:
CENSUS_VARS = {
    'B01003_001E': 'population',          # total population
    'B19013_001E': 'median_income',       # median household income ($)
    'B08301_001E': 'commuters_total',     # workers 16+
    'B08301_018E': 'commuters_bike',      # workers who bike to work
    'B25044_001E': 'households_total',    # occupied housing units
    'B25044_003E': 'no_vehicle_owner',    # owner households with no vehicle
    'B25044_010E': 'no_vehicle_renter',   # renter households with no vehicle
}
params_census = {
    'get': 'NAME,' + ','.join(CENSUS_VARS),
    'for': 'place:*',                         # every city/town...
    'in': 'state:*',                          # ...in every state
    'key': CENSUS_API_KEY,
}

try:
    response = requests.get('https://api.census.gov/data/2024/acs/acs5', params=params_census, timeout=60)
except requests.RequestException as e:
    raise RuntimeError(f'Census request failed ({type(e).__name__})') from None
if response.status_code != 200:
    raise RuntimeError(f'Census API returned {response.status_code}: {response.text[:300]}')

census_raw = response.json()          # list of rows; the first row is the header

## 5. Clean the Census data

- **geoid** = state code + place code. This is the unique ID for each city.
- Numbers arrive as text, so we convert them. **Negative values (like `-666666666`) are Census codes for "no data", so they become `NaN`.**
- `"Chicago city, Illinois"` becomes `city = "Chicago"` and `state = "Illinois"`.
- `pct_bike_commute = commuters_bike / commuters_total`
- `pct_no_vehicle = (no_vehicle_owner + no_vehicle_renter) / households_total`

In [ ]:
raw = pd.DataFrame(census_raw[1:], columns=census_raw[0]).rename(columns=CENSUS_VARS)
num_cols = list(CENSUS_VARS.values())
raw[num_cols] = raw[num_cols].apply(pd.to_numeric, errors='coerce')
raw[num_cols] = raw[num_cols].mask(raw[num_cols] < 0)          # Census "no data" codes -> NaN

census_df = pd.DataFrame({'geoid': raw['state'] + raw['place']})
census_df[['city', 'state']] = raw['NAME'].str.rsplit(', ', n=1, expand=True)
# shortcut: strips only common place types; odd names like "Nashville-Davidson metropolitan government (balance)" stay as-is
census_df['city'] = census_df['city'].str.replace(r'\s+(city|town|village|borough|CDP|municipality)$', '', regex=True)
census_df['population'] = raw['population']
census_df['median_income'] = raw['median_income']
census_df['pct_bike_commute'] = (raw['commuters_bike'] / raw['commuters_total']).round(4)
census_df['pct_no_vehicle'] = ((raw['no_vehicle_owner'] + raw['no_vehicle_renter']) / raw['households_total']).round(4)

pct = census_df[['pct_bike_commute', 'pct_no_vehicle']]
assert pct.stack().dropna().between(0, 1).all(), 'a share is outside 0-1'

census_df = census_df.sort_values('population', ascending=False).reset_index(drop=True)
census_df.head(10)

## 6. Save the two CSV files

In [ ]:
citybike_df.to_csv(DATA_DIR / 'citybike_networks_us.csv', index=False)
census_df.to_csv(DATA_DIR / 'census_places.csv', index=False)
print(f'citybike_networks_us.csv: {len(citybike_df):,} rows')
print(f'census_places.csv: {len(census_df):,} rows')